In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine

In [2]:
fund_master = pd.read_csv('fund_master.csv')

In [3]:
aum_by_fund_house = pd.read_csv('aum_by_fund_house.csv')
portfolio_holdings = pd.read_csv('portfolio_holdings.csv')
benchmark_indices = pd.read_csv('benchmark_indices.csv')
category_inflows = pd.read_csv('category_inflows.csv')
industry_folio_count = pd.read_csv('industry_folio_count.csv')
investor_transactions = pd.read_csv('investor_transactions.csv')
monthly_sip_inflows = pd.read_csv('monthly_sip_inflows.csv')
nav_history = pd.read_csv('nav_history.csv')
scheme_performance = pd.read_csv('scheme_performance.csv')

In [4]:
#check
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [5]:
nav_history['date'] = pd.to_datetime(
    nav_history['date']
)

In [6]:
nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [7]:
nav_history = nav_history.sort_values(
    ['amfi_code','date']
)

In [8]:
nav_history.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [9]:
nav_history['nav'] = nav_history.groupby(
    'amfi_code'
)['nav'].ffill()

In [10]:
#remove duplicate
nav_history = nav_history.drop_duplicates()

In [11]:
nav_history.shape

(46000, 3)

In [12]:
#remove invalid nav
nav_history = nav_history[
    nav_history['nav'] > 0
]

In [13]:
nav_history.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/processed/clean_nav.csv',
    index=False
)

In [14]:
#investor_transactions['investor_transactions_date'] = pd.to_datetime(
#    investor_transactions ['investor_transactions_date']
#)

In [15]:
print(investor_transactions.columns)

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')


In [16]:
# Clean Investor Transactions
investor_transactions['transaction_date'] = pd.to_datetime(
    investor_transactions['transaction_date']
)

In [17]:
#Standardize Transaction Type
investor_transactions['transaction_type'] = (
    investor_transactions['transaction_type']
    .str.strip()
    .str.title()
)

In [18]:
investor_transactions['transaction_type'].value_counts()

transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [19]:
#remove invalid amounts
investor_transactions = investor_transactions[
    investor_transactions['amount_inr'] > 0
]

In [20]:
#check KYC
investor_transactions['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [21]:
investor_transactions.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone\data/processed/clean_transactions.csv',
    index=False
)

In [22]:
#Clean Scheme Performance Dataset
#convert columns
scheme_performance['return_1yr_pct'] = pd.to_numeric(
    scheme_performance['return_1yr_pct']
)

In [23]:
scheme_performance['return_3yr_pct'] = pd.to_numeric(
    scheme_performance['return_3yr_pct']
)
scheme_performance['return_5yr_pct'] = pd.to_numeric(
    scheme_performance['return_5yr_pct']
)
scheme_performance['sharpe_ratio'] = pd.to_numeric(
    scheme_performance['sharpe_ratio']
)
scheme_performance['alpha'] = pd.to_numeric(
    scheme_performance['alpha']
)
scheme_performance['beta'] = pd.to_numeric(
    scheme_performance['beta']
)

In [24]:
#to find negative sharpe
scheme_performance[
    scheme_performance['sharpe_ratio'] < 0
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [25]:
#expence ratio check
scheme_performance[
    (scheme_performance['expense_ratio_pct'] < 0.1)
    |
    (scheme_performance['expense_ratio_pct'] > 2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [26]:
scheme_performance.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/processed/clean_performance.csv',
    index=False
)

In [27]:
#create sqlite database
engine = create_engine(
    r'sqlite:///C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/db/bluestock_mf.db'
)

In [28]:
#fund_master.to_sql(
#   'dim_fund',
#    engine,
#  if_exists='replace',
#    index=False
#)

In [29]:
import os

os.makedirs(r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/db', exist_ok=True)

In [30]:
from sqlalchemy import create_engine

engine = create_engine(
    r'sqlite:///C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/db/bluestock_mf.db'
)

In [31]:
#fund table
fund_master.to_sql(
    'dim_fund',
    engine,
    if_exists='replace',
    index=False
)

40

In [32]:
#nav table
nav_history.to_sql(
    'fact_nav',
    engine,
    if_exists='replace',
    index=False
)

46000

In [33]:
# investor transaction table
investor_transactions.to_sql(
    'fact_transactions',
    engine,
    if_exists='replace',
    index=False
)

32778

In [34]:
# schema performance table
scheme_performance.to_sql(
    'fact_performance',
    engine,
    if_exists='replace',
    index=False
)

40

In [35]:
conn = sqlite3.connect(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/db/bluestock_mf.db'
)

In [36]:
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

,name
0,dim_fund
1,fact_nav
2,fact_transactions
3,fact_performance


In [37]:
#SQL queries
#1 top funds
query = """
SELECT *
FROM dim_fund
LIMIT 5
"""

pd.read_sql(query, conn)

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [38]:
#2 transaction count
query = """
SELECT transaction_type,
COUNT(*) total
FROM fact_transactions
GROUP BY transaction_type
"""

pd.read_sql(query, conn)

,transaction_type,total
0,Lumpsum,8095
1,Redemption,4967
2,Sip,19716


In [39]:
#3 transaction by state
query = """
SELECT state,
COUNT(*) total
FROM fact_transactions
GROUP BY state
ORDER BY total DESC
"""

pd.read_sql(query, conn)

,state,total
0,Punjab,2965
1,Madhya Pradesh,2931
2,Tamil Nadu,2806
3,Gujarat,2780
4,West Bengal,2748
5,Haryana,2736
6,Telangana,2718
7,Uttar Pradesh,2695
8,Delhi,2677
9,Karnataka,2621


In [40]:
#4 average NAV
query = """
SELECT AVG(nav)
FROM fact_nav
"""

pd.read_sql(query, conn)

,AVG(nav)
0,269.570265


In [41]:
#5 Top 10 Funds by 3-Year Return
query = """
SELECT
    amfi_code,
    return_3yr_pct
FROM fact_performance
ORDER BY return_3yr_pct DESC
LIMIT 10;
"""

top_returns = pd.read_sql(query, conn)
top_returns

,amfi_code,return_3yr_pct
0,119598,23.39
1,119599,23.14
2,101207,22.38
3,119095,20.98
4,118634,20.15
5,149324,20.08
6,120842,18.23
7,120505,18.08
8,149323,17.16
9,100033,16.58


In [42]:
#6 Average Return by Fund Category
query = """
SELECT
    f.category,
    ROUND(AVG(p.return_3yr_pct),2) AS avg_return
FROM fact_performance p
JOIN dim_fund f
ON p.amfi_code = f.amfi_code
GROUP BY f.category;
"""

avg_category_return = pd.read_sql(query, conn)
avg_category_return

,category,avg_return
0,Debt,6.29
1,Equity,15.46


In [43]:
#7 Top States by Investment Amount
query = """
SELECT
    state,
    SUM(amount_inr) AS total_investment
FROM fact_transactions
GROUP BY state
ORDER BY total_investment DESC;
"""

state_investment = pd.read_sql(query, conn)
state_investment

,state,total_investment
0,Punjab,315780459
1,Tamil Nadu,315177237
2,Madhya Pradesh,308312493
3,Rajasthan,298645822
4,Gujarat,298358940
5,West Bengal,297182514
6,Telangana,290219284
7,Delhi,289633404
8,Uttar Pradesh,285368873
9,Haryana,279634354


In [44]:
#8 KYC Status Distribution
query = """
SELECT
    kyc_status,
    COUNT(*) AS investor_count
FROM fact_transactions
GROUP BY kyc_status;
"""

kyc_distribution = pd.read_sql(query, conn)
kyc_distribution

,kyc_status,investor_count
0,Pending,2632
1,Verified,30146


In [45]:
#9 Investment by City Tier
query = """
SELECT
    city_tier,
    SUM(amount_inr) AS total_amount
FROM fact_transactions
GROUP BY city_tier;
"""

city_tier_data = pd.read_sql(query, conn)
city_tier_data

,city_tier,total_amount
0,B30,1202325640
1,T30,2319254790


In [46]:
#10 Top 10 Funds by Sharpe Ratio
query = """
SELECT
    amfi_code,
    sharpe_ratio
FROM fact_performance
ORDER BY sharpe_ratio DESC
LIMIT 10;
"""

top_sharpe = pd.read_sql(query, conn)
top_sharpe

,amfi_code,sharpe_ratio
0,120507,7.68
1,120844,6.18
2,101208,5.14
3,100025,1.84
4,119120,1.52
5,118636,1.33
6,100016,1.06
7,148567,1.06
8,120504,1.03
9,118632,1.00


In [47]:
# dataframe create
dictionary = pd.DataFrame({
    "Column":[
        "amfi_code",
        "scheme_name",
        "nav",
        "transaction_type"
    ],
    "Description":[
        "Fund code",
        "Fund Name",
        "Net Asset Value",
        "Type of investment"
    ]
})

In [48]:
dictionary.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/report/data_dictionary.csv',
    index=False
)